# 18 — Aspect visuals (coast bands + year slices)

Charts over the Claude silver labels (`TOPIC_ASPECTS` + `REVIEW_TOPICS`):

| # | Chart | Slice | Style |
|---|-------|-------|-------|
| 1 | 5 `key_aspect` bars, stacked by **3 distance bands** (3 colors) | coast | horizontal stacked bar |
| 2 | top-5 `sub_aspect` underneath each `key_aspect`, stacked by band | coast | horizontal stacked bar × 5 panels |
| 3 | `key_aspect` share over **years** (5 lines) | year | line graph |
| 4 | top-5 `sub_aspect` per aspect over years | year | line graph × 5 panels |

Weighted counting: a review in a topic labeled 0.6 service / 0.4 facility contributes
0.6 and 0.4 respectively (`weight × n_reviews`).

**Prerequisite:** labels must exist — run `notebooks/17_silver_label_test.ipynb` then
`src/llm_label_submit.py` + `src/llm_label_retrieve.py` first.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append("../src")

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import llm_label as ll

IMG_DIR = Path("../img")
IMG_DIR.mkdir(exist_ok=True)

ASPECT_ORDER = ["facility", "amenity", "service", "experience", "loyalty"]
ASPECT_COLORS = {
    "facility":   "#4878a8",
    "amenity":    "#e49444",
    "service":    "#6a9f58",
    "experience": "#a87ca8",
    "loyalty":    "#d1605e",
}
BAND_ORDER = ["Beachfront (<0.1 km)", "Near-coast (0.1\u20130.5 km)", "Inland (\u22650.5 km)"]
BAND_COLORS = {  # sea -> land gradient
    BAND_ORDER[0]: "#0277bd",
    BAND_ORDER[1]: "#26a69a",
    BAND_ORDER[2]: "#8d6e63",
}
TITLE_KW = dict(fontsize=16, fontweight="bold", color="#36648B", loc="left")

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 14, "axes.labelsize": 12,
                     "xtick.labelsize": 11, "ytick.labelsize": 11})

## Load labeled data
One row per (run, topic, aspect) with its weighted review count.

In [ ]:
con = duckdb.connect(str(ll.DB_PATH), read_only=True)

df = con.execute("""
    SELECT rt.run_id, rt.topic_id, ta.key_aspect, ta.weight, ta.sentiment,
           ta.sub_aspects, COUNT(*) AS n_reviews
    FROM REVIEW_TOPICS rt
    JOIN TOPIC_ASPECTS ta
      ON rt.run_id = ta.run_id AND rt.topic_id = ta.topic_id
    WHERE rt.topic_id != -1 AND ta.key_aspect != 'other'
    GROUP BY ALL
""").df()
con.close()

if df.empty:
    raise RuntimeError("TOPIC_ASPECTS is empty - run the labeling batch first "
                       "(notebook 17, then llm_label_submit.py / llm_label_retrieve.py).")

df["weighted"] = df["weight"] * df["n_reviews"]
df["language"] = df["run_id"].str[-2:]

coast = df[df["run_id"].str.startswith("coast_band_")].copy()
coast["band"] = (coast["run_id"].str.extract(r"coast_band_([ABC])")[0]
                 .map(dict(zip(["A", "B", "C"], BAND_ORDER))))

year = df[df["run_id"].str.startswith("year_")].copy()
year["year"] = year["run_id"].str.extract(r"year_(\d{4})")[0].astype(int)

print(f"labeled aspect rows: coast={len(coast)}, year={len(year)}")
print("labeled runs:", sorted(df.run_id.unique()))

## Chart 1 — Reviews per key_aspect, stacked by distance band
5 horizontal bars (one per aspect), 3 colors (one per band), value labels per segment —
same layout as the ggplot stacked-`geom_col` + `coord_flip` reference.

In [ ]:
pivot = (coast.groupby(["key_aspect", "band"])["weighted"].sum()
         .unstack("band")
         .reindex(index=ASPECT_ORDER, columns=BAND_ORDER)
         .fillna(0))

fig, ax = plt.subplots(figsize=(12, 6))
left = pd.Series(0.0, index=pivot.index)
for band in BAND_ORDER:
    vals = pivot[band]
    ax.barh(pivot.index, vals, left=left, height=0.55,
            color=BAND_COLORS[band], label=band)
    for aspect, v, l in zip(pivot.index, vals, left):
        if v > pivot.to_numpy().sum() * 0.01:  # skip labels on tiny slivers
            ax.text(l + v / 2, aspect, f"{v:,.0f}",
                    ha="center", va="center", color="white",
                    fontsize=10, fontweight="bold")
    left += vals

ax.invert_yaxis()
ax.set_title("Aspect mentions per distance band (weighted reviews)", **TITLE_KW)
ax.set_xlabel("Weighted review count")
ax.set_ylabel("key_aspect")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
fig.tight_layout()
fig.savefig(IMG_DIR / "18_coast_aspect_stacked.png", dpi=150, bbox_inches="tight")
plt.show()

### Same chart as % within each band
Bands have very different sizes (Inland ≫ Beachfront), so the share view answers
“do beachfront guests talk about different things?” better than raw counts.

In [ ]:
share = (coast.groupby(["band", "key_aspect"])["weighted"].sum()
         .groupby(level="band").transform(lambda s: 100 * s / s.sum())
         .unstack("key_aspect")
         .reindex(index=BAND_ORDER, columns=ASPECT_ORDER)
         .fillna(0))

fig, ax = plt.subplots(figsize=(12, 4.5))
left = pd.Series(0.0, index=share.index)
for aspect in ASPECT_ORDER:
    vals = share[aspect]
    ax.barh(share.index, vals, left=left, height=0.55,
            color=ASPECT_COLORS[aspect], label=aspect)
    for band, v, l in zip(share.index, vals, left):
        if v > 3:
            ax.text(l + v / 2, band, f"{v:.0f}%", ha="center", va="center",
                    color="white", fontsize=10, fontweight="bold")
    left += vals

ax.invert_yaxis()
ax.set_title("Aspect share within each distance band", **TITLE_KW)
ax.set_xlabel("% of weighted reviews in band")
ax.set_xlim(0, 100)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=5, frameon=False)
fig.tight_layout()
fig.savefig(IMG_DIR / "18_coast_aspect_share.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 2 — Top-5 sub_aspects underneath each key_aspect (coast)
One panel per aspect; within it, that aspect's top-5 `sub_aspect` bars, stacked by the 3 bands.

In [ ]:
subs = coast.copy()
subs["sub_aspect"] = subs["sub_aspects"].apply(json.loads)
subs = subs.explode("sub_aspect").dropna(subset=["sub_aspect"])

fig, axes = plt.subplots(len(ASPECT_ORDER), 1, figsize=(12, 3.2 * len(ASPECT_ORDER)),
                         sharex=False)

for ax, aspect in zip(axes, ASPECT_ORDER):
    d = subs[subs["key_aspect"] == aspect]
    if d.empty:
        ax.set_visible(False)
        continue
    top5 = d.groupby("sub_aspect")["weighted"].sum().nlargest(5).index
    p = (d[d["sub_aspect"].isin(top5)]
         .groupby(["sub_aspect", "band"])["weighted"].sum()
         .unstack("band").reindex(index=top5, columns=BAND_ORDER).fillna(0))

    left = pd.Series(0.0, index=p.index)
    for band in BAND_ORDER:
        ax.barh(p.index, p[band], left=left, height=0.6,
                color=BAND_COLORS[band], label=band)
        left += p[band]
    for sub_name, total in left.items():
        ax.text(total, sub_name, f" {total:,.0f}", va="center", fontsize=9)

    ax.invert_yaxis()
    ax.set_title(aspect.upper(), fontsize=13, fontweight="bold",
                 color=ASPECT_COLORS[aspect], loc="left")
    ax.set_xlabel("")

axes[0].legend(loc="lower right", fontsize=9, frameon=False)
fig.suptitle("Top-5 sub_aspects per key_aspect — coast bands",
             fontsize=16, fontweight="bold", color="#36648B", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.98])
fig.savefig(IMG_DIR / "18_coast_subaspects.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 3 — Aspect share over years (line graph)
5 lines, one per key_aspect; y = % of weighted reviews within the year (en + vi combined).

In [ ]:
if year.empty:
    print("No year runs labeled yet - run llm_label_submit.py --all first.")
else:
    yshare = (year.groupby(["year", "key_aspect"])["weighted"].sum()
              .groupby(level="year").transform(lambda s: 100 * s / s.sum())
              .unstack("key_aspect").reindex(columns=ASPECT_ORDER))

    fig, ax = plt.subplots(figsize=(12, 6))
    for aspect in ASPECT_ORDER:
        if aspect in yshare:
            ax.plot(yshare.index, yshare[aspect], marker="o", linewidth=2.5,
                    color=ASPECT_COLORS[aspect], label=aspect)

    ax.set_title("Aspect share over years (2018\u20132024)", **TITLE_KW)
    ax.set_xlabel("Year")
    ax.set_ylabel("% of weighted reviews")
    ax.set_xticks(sorted(year["year"].unique()))
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=5, frameon=False)
    fig.tight_layout()
    fig.savefig(IMG_DIR / "18_year_aspect_lines.png", dpi=150, bbox_inches="tight")
    plt.show()

## Chart 4 — Top-5 sub_aspects per key_aspect over years (line graphs)
One panel per aspect; lines = that aspect's overall top-5 `sub_aspect`s, y = weighted reviews.

In [ ]:
if year.empty:
    print("No year runs labeled yet - run llm_label_submit.py --all first.")
else:
    ysubs = year.copy()
    ysubs["sub_aspect"] = ysubs["sub_aspects"].apply(json.loads)
    ysubs = ysubs.explode("sub_aspect").dropna(subset=["sub_aspect"])

    fig, axes = plt.subplots(len(ASPECT_ORDER), 1,
                             figsize=(12, 3.4 * len(ASPECT_ORDER)), sharex=True)
    xticks = sorted(year["year"].unique())

    for ax, aspect in zip(axes, ASPECT_ORDER):
        d = ysubs[ysubs["key_aspect"] == aspect]
        if d.empty:
            ax.set_visible(False)
            continue
        top5 = d.groupby("sub_aspect")["weighted"].sum().nlargest(5).index
        p = (d[d["sub_aspect"].isin(top5)]
             .groupby(["year", "sub_aspect"])["weighted"].sum()
             .unstack("sub_aspect").reindex(columns=top5).fillna(0))
        for i, sub_name in enumerate(top5):
            ax.plot(p.index, p[sub_name], marker="o", linewidth=2,
                    alpha=1 - 0.15 * i, color=ASPECT_COLORS[aspect], label=sub_name)
        ax.set_title(aspect.upper(), fontsize=13, fontweight="bold",
                     color=ASPECT_COLORS[aspect], loc="left")
        ax.grid(axis="y", alpha=0.3)
        ax.legend(fontsize=9, ncol=2, frameon=False)

    axes[-1].set_xticks(xticks)
    axes[-1].set_xlabel("Year")
    fig.suptitle("Top-5 sub_aspects per key_aspect over years",
                 fontsize=16, fontweight="bold", color="#36648B", x=0.01, ha="left")
    fig.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig(IMG_DIR / "18_year_subaspect_lines.png", dpi=150, bbox_inches="tight")
    plt.show()

All figures are saved to `img/` (`18_coast_aspect_stacked.png`, `18_coast_aspect_share.png`,
`18_coast_subaspects.png`, `18_year_aspect_lines.png`, `18_year_subaspect_lines.png`).